In [3]:
import sys, os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error

torch.manual_seed(42)
np.random.seed(42)

def load_data():
    X, y = load_wine(return_X_y=True)
    X = (X - X.mean(0)) / X.std(0)

    return train_test_split(
        torch.FloatTensor(X),
        torch.LongTensor(y),
        test_size=0.2,
        random_state=42,
        stratify=y
    )

def build():
    return nn.Sequential(
        nn.Linear(13, 64),
        nn.ReLU(),
        nn.Dropout(0.15),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 3)
    )

def train(model, X, y):
    optimizer = optim.AdamW(model.parameters(), lr=0.005, weight_decay=1e-4)
    lFN = nn.CrossEntropyLoss()

    dataset = torch.utils.data.TensorDataset(X, y)
    loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)

    model.train()
    for _ in range(300):
        for xb, yb in loader:
            optimizer.zero_grad()
            l = lFN(model(xb), yb)
            l.backward()
            optimizer.step()

def eval(model, X, y):
    model.eval()
    with torch.no_grad():
        pred = torch.argmax(model(X), dim=1)

    return {
        "acc": accuracy_score(y.cpu().numpy(), pred.cpu().numpy()),
        "mse": mean_squared_error(y.cpu().numpy(), pred.cpu().numpy())
    }

def save(model, metrics):
    os.makedirs("outputFile", exist_ok=True)
    torch.save(model.state_dict(), "outputFile/wine_mlp.pt")
    with open("outputFile/wine_metrics.txt", "w") as f:
        f.write(str(metrics))

if __name__ == "__main__":
    X_train, X_val, y_train, y_val = load_data()
    model = build()

    train(model, X_train, y_train)

    vm = eval(model, X_val, y_val)
    save(model, vm)

    print(vm)

    success = vm["acc"] > 0.90
    if success:
        print("pass")
        sys.exit(0)
    else:
        print("fail")
        sys.exit(1)

{'acc': 0.9722222222222222, 'mse': 0.027777777777777776}
pass


SystemExit: 0

/Users/Varun/Library/Python/3.9/lib/python/site-packages/IPython/core/interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
